In [2]:
import numpy as np
import gymnasium as gym

env = gym.make("CartPole-v1")

n_bins = 20
bins = [
    np.linspace(-4.8, 4.8, n_bins),
    np.linspace(-4.0, 4.0, n_bins),
    np.linspace(-0.418, 0.418, n_bins),
    np.linspace(-4.0, 4.0, n_bins)
]

def discretize_state(state):
    idxs = []
    for i, value in enumerate(state):
        idx = np.digitize(value, bins[i])
        idx = np.clip(idx, 0, n_bins)
        idxs.append(idx)
    return tuple(idxs)

q_table = np.zeros((n_bins + 1, n_bins + 1, n_bins + 1, n_bins + 1, env.action_space.n))

alpha = 0.1
gamma = 0.99
epsilon = 1.0

for episode in range(3000):
    state, _ = env.reset()
    state = discretize_state(state)
    done = False

    while not done:
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[state])

        next_state, reward, terminated, truncated, _ = env.step(action)
        next_state = discretize_state(next_state)
        done = terminated or truncated

        q_table[state][action] += alpha * (
            reward + gamma * np.max(q_table[next_state]) - q_table[state][action]
        )

        state = next_state

    epsilon = max(0.05, epsilon * 0.999)